# 3D-VGA — ablations and analysis

Runs every audio experiment from one notebook. Built for **Run All**: nothing that can't run on this machine raises, it just skips, and an experiment whose outputs exist is not repeated.

| phase | what it does | needs |
|---|---|---|
| **A — training arms** | one run per ablation arm, each evaluated as it finishes | GPU |
| **B — evaluation only** | controls and knockouts on the trained model; trains nothing | GPU, minutes each |
| **C — sweeps** | one setting varied at a time; off by default | GPU |
| **D — tables and analyses** | reads files off disk | CPU, seconds |
| **E — robustness** | unseen speakers, noise, ASR error rate | CPU / GPU |

Every training run fits only the acoustic branch and the fusion head (2.1 M parameters). Proposals come from `cached_scenes/` and speech features from `data_audio/feat/`, so no detector and no speech encoder runs. Before this: `run_audio_encode.ipynb`.

## 1 — configuration

The only cell you need to edit.

In [ ]:
# USER CONFIGURATION

DEVICE       = "auto"     # "auto" | "cpu" | "cuda"   -- auto probes for a working GPU
GPU_ID       = "0"
FORCE_GPU    = False      # run GPU experiments without a usable GPU (very slow)
OVERRIDE_EXP = False      # True re-runs experiments whose output files already exist
MIN_BATCH_SIZE = 1        # batch size substituted when FORCE_GPU falls back to CPU
QUICK_TEST   = False      # True: 2 scenes, 2 epochs, runs tagged QUICK-, data_audio/_quick.
                          #       Checks the plumbing in minutes; its numbers mean nothing.

USE_CACHED_SCENES  = True
CACHED_SCENES_ROOT = "cached_scenes"
ENCODER            = "wav2vec2"     # which feature cache to read

# TRAINING -- applies to every arm and sweep point, exported as VGA_* by the apply cell.

EPOCH        = 30
BATCH_SIZE   = 8          # scenes per batch
LANG_NUM_MAX = 8          # spoken descriptions per scene per batch
LR           = 1e-3
SEED         = 42
VAL_STEP     = 2000       # validate every N ITERATIONS; a checkpoint is saved only then
VERBOSE      = 50
NUM_WORKERS  = 4
EXTRA_ARGS   = ["--use_color", "--use_normal", "--coslr"]    # MUST match the scene cache
CHECKPOINT_FILE = "model_criteria_25.pth"     # checkpoint that evaluation scores

# PHASE A -- training arms. One run each; each differs from "spot" in exactly one thing.

RUN_ARMS = {
    # core: the baselines, the central claim, and the two acoustic-side components
    "spot":             True,    # A1  proposed model                 (reference arm)
    "concat":           True,    # A2  AP-Refer's fusion, reproduced
    "xattn":            True,    # A2  attentive fusion, no transport
    "spot-softmax":     True,    # A3  transport constraint removed   (the central claim)
    "spot-norsk":       True,    # A4  fixed distance prior instead of the relation kernel
    "spot-noprosody":   True,    # A4  no prosody stream
    # extended: the remaining leave-one-out arms
    "spot-k2":          False,   # A6  only two slots
    "spot-unidir":      False,   # A6  slots never read the proposals
    "spot-fixedroles":  False,   # A6  fixed slot indices instead of role election
    "spot-nograph":     False,   # A6  no spatial graph
    "spot-nogate":      False,   # A6  plain residual instead of the read-out gate
    "spot-noframes":    False,   # A6  no residual pass over raw frames
    "spot-nocopypaste": False,   # A6  no proposal copy-paste
    "spot-notimeprior": False,   # A6  no time prior on the slots
    "spot-nospecaug":   False,   # A6  no SpecAugment
}

RUN_TRAIN_SEEDS = True           # A5  the proposed model under more seeds
SEEDS = [1, 2]                   #     extra seeds; seed 42 is the "spot" arm itself

# PHASE B -- evaluation only. Minutes each; needs a trained "spot" run.

RUN_EVAL_ARMS = True             # B1  score any trained arm that has no scores yet
RUN_CONTROLS  = True             # B2  is the model listening? (silence, mismatched audio, ...)
RUN_KNOCKOUT  = True             # B3  one component switched off at test time

# PHASE C -- sweeps. Each point is a training run; the default point is the "spot" arm.

RUN_SWEEPS = {
    "slots":              False,   # number of role slots
    "depth":              False,   # number of fusion layers
    "graph_k":            False,   # neighbours in the spatial graph
    "audio_cls":          False,   # weight of the audio classification loss
    "prosody_channels":   False,   # which prosodic cue group
    "ot_weight":          False,   # initial weight of the transport logit
    "sinkhorn":           False,   # entropic temperature and iterations
    "slot_shaping":       False,   # slot entropy / orthogonality weights
    "role_separation":    False,   # role separation weight
    "target_contrastive": False,   # contrastive weight
    # these three re-encode the WAVs once per point: minutes and several GB each
    "encoder_layer":      False,
    "encoder":            False,
    "frames":             False,
}
# Lighter grids for the single-flag sweeps. Delete an entry to run that sweep's full grid.
SWEEP_VALUES = {
    "slots":            [4, 8],
    "depth":            [1, 2],
    "graph_k":          [0, 10],
    "prosody_channels": ["energy", "pitch", "spectral"],
    "ot_weight":        [0.0, 1.0],
    "encoder":          ["wavlm"],
}

# PHASE D -- tables and analyses. CPU only; reads files off disk.

RUN_TABLES   = True              # D1  ablation, intervention, seed and sweep tables
RUN_ANALYSES = True              # D2  slot roles, transport sharpness, error taxonomy, ...
RUN_COST     = True              # D3  parameters and latency

# PHASE E -- robustness.

RUN_SPEAKER_OOD = True           # E1  seen vs held-out voices -- CPU
RUN_NOISE       = False          # E2  noisy audio -- GPU, re-encodes val once per SNR
NOISE_SNR       = [20, 10, 5, 0]
RUN_ASR_WER     = False          # E3  transcribe val and report word-error rate -- GPU

WAV_ROOT = ""                    # folder holding wav/ if it is not data_audio/. On Colab the
                                 # Drive copy written by run_audio_encode.ipynb is found by itself.

# GOOGLE DRIVE -- Colab only
MIRROR_TO_DRIVE = True
DRIVE_FOLDER    = "3DVGA"        # under MyDrive/

QUICK_SCENES = 2

## 2 — environment

Probes the GPU by launching a kernel — `torch.cuda.is_available()` alone can report `True` on an unsupported card.

In [ ]:
import glob as _glob
import importlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

REPO = Path.cwd()
RESULTS = {}

if not (REPO / "scripts" / "audio_train.py").is_file():
    raise SystemExit(f"Run this notebook from the repository root. Current cwd: {REPO}")

# lib/config.py lists this directory at import time; an empty one is enough for the steps
# that never read a scan.
(REPO / "data" / "scannet" / "scans").mkdir(parents=True, exist_ok=True)

IN_COLAB = importlib.util.find_spec("google.colab") is not None

GPU_USABLE, GPU_NAME, GPU_NOTE = False, None, ""
try:
    import torch
    if torch.cuda.is_available():
        GPU_NAME = torch.cuda.get_device_name(0)
        major, minor = torch.cuda.get_device_capability(0)
        try:
            _probe = torch.randn(8, 8, device="cuda")
            (_probe @ _probe).sum().item()
            GPU_USABLE = True
            GPU_NOTE = f"{GPU_NAME} (sm_{major}{minor})"
        except Exception as error:
            arch = getattr(torch.cuda, "get_arch_list", lambda: ["?"])()
            GPU_NOTE = (f"{GPU_NAME} (sm_{major}{minor}) present but NO KERNEL RUNS -- "
                        f"this torch ships {arch}. "
                        f"{type(error).__name__}: {str(error).splitlines()[0][:80]}")
    else:
        GPU_NOTE = "no CUDA device visible to torch"
except Exception as error:
    GPU_NOTE = f"torch unavailable: {error}"

if DEVICE == "auto":
    DEVICE = "cuda" if GPU_USABLE else "cpu"
elif DEVICE == "cuda" and not GPU_USABLE:
    print("[warn] DEVICE='cuda' but no kernel launches here.")

print("=" * 88)
print(f"environment  : {'Google Colab' if IN_COLAB else 'local machine'}")
print(f"python       : {sys.version.split()[0]}")
print(f"GPU          : {'USABLE -- ' + GPU_NOTE if GPU_USABLE else 'UNUSABLE -- ' + GPU_NOTE}")
print(f"device       : {DEVICE}")
print(f"quick test   : {QUICK_TEST}")
print("=" * 88)

if not GPU_USABLE and not FORCE_GPU:
    print("GPU experiments will be SKIPPED. Set FORCE_GPU=True to run them on CPU "
          f"at batch size {MIN_BATCH_SIZE} (expect many hours).")

## 3 — packages

Colab only. The audio model needs no `pyg-lib`: its spatial graph is a dense kNN.

In [ ]:
EXTRA_PACKAGES = ["h5py", "transformers"]
REQUIRED_MODULES = ["torch", "numpy", "h5py", "easydict", "plyfile", "tensorboardX", "tqdm"]

def _pip(*args):
    print("$ pip install " + " ".join(args))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

def ensure_packages(force=False):
    if not (IN_COLAB or force):
        print("not on Colab -- skipping installs. Pass force=True to run them anyway.")
        return
    if (REPO / "req" / "requirements.txt").is_file():
        _pip("-r", str(REPO / "req" / "requirements.txt"))
    _pip(*EXTRA_PACKAGES)

ensure_packages()

print("\n-- packages --")
MISSING_PACKAGES = []
for module in REQUIRED_MODULES:
    try:
        loaded = importlib.import_module(module)
        print(f"  {module:18s} OK    {getattr(loaded, '__version__', '')}")
    except Exception:
        MISSING_PACKAGES.append(module)
        print(f"  {module:18s} --    MISSING")
if MISSING_PACKAGES:
    print(f"\n  install first: pip install {' '.join(MISSING_PACKAGES)}")

## 4 — Google Drive

Colab runtimes are disposable — `outputs/` is copied to Drive after every experiment, and the feature caches and earlier runs are brought back on a fresh runtime.

In [ ]:
# Off Colab this whole cell is inert, so the notebook stays runnable locally.

DRIVE_ROOT = None

def mount_drive():
    """Mount Drive and return the destination directory, or None if unavailable."""
    global DRIVE_ROOT
    if not (MIRROR_TO_DRIVE and IN_COLAB):
        print("Drive off"
              + ("" if MIRROR_TO_DRIVE else " (MIRROR_TO_DRIVE=False)")
              + ("" if IN_COLAB else " -- not running on Colab, everything stays local"))
        return None
    try:
        from google.colab import drive
        mount = Path("/content/drive")
        if not (mount / "MyDrive").is_dir():
            drive.mount(str(mount))
        DRIVE_ROOT = mount / "MyDrive" / DRIVE_FOLDER
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print(f"Drive mounted -> {DRIVE_ROOT}")
    except Exception as error:
        # A failed mount must not take the run with it.
        DRIVE_ROOT = None
        print(f"Drive not mounted ({error}). The run continues; results stay on this "
              f"machine only and will be lost when the runtime recycles.")
    return DRIVE_ROOT

def copy_newer(source, target, quiet=False):
    """Copy a file or a tree, skipping files the target already has up to date."""
    source, target = Path(source), Path(target)
    if not source.exists():
        return 0
    files = [source] if source.is_file() else [p for p in source.rglob("*") if p.is_file()]
    copied = 0
    for path in files:
        if "tensorboard" in path.parts or path.suffix == ".tmp":
            continue
        dest = target if source.is_file() else target / path.relative_to(source)
        try:
            if dest.exists() and dest.stat().st_mtime >= path.stat().st_mtime \
                    and dest.stat().st_size == path.stat().st_size:
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, dest)
            copied += 1
        except Exception as error:
            if not quiet:
                print(f"  could not copy {path.name}: {error}")
    return copied

def mirror_to_drive(quiet=False):
    """Copy outputs/ to Drive. Cheap to call repeatedly: only newer files are copied."""
    if DRIVE_ROOT is None:
        return 0
    copied = copy_newer(REPO / "outputs", DRIVE_ROOT / "outputs", quiet=quiet)
    if not quiet:
        print(f"mirrored {copied} new/changed file(s) to {DRIVE_ROOT}/outputs")
    return copied

def restore_from_drive(*relative):
    """Bring files back from Drive after a fresh runtime, where they are missing here."""
    if DRIVE_ROOT is None:
        return
    for rel in relative:
        copied = copy_newer(DRIVE_ROOT / rel, REPO / rel, quiet=True)
        if copied:
            print(f"restored {copied} file(s) from Drive: {rel}")

mount_drive()

if not QUICK_TEST:
    restore_from_drive(f"data_audio/feat/{ENCODER}/train.h5", f"data_audio/feat/{ENCODER}/val.h5",
                       "data_audio/manifest_train.json", "data_audio/manifest_val.json",
                       "outputs")

## 5 — apply the configuration

The runner scripts read their shared settings from `VGA_<NAME>` environment variables, exported here — no source file is rewritten. Re-run this cell after editing the config cell.

In [ ]:
USE_CPU = (DEVICE == "cpu") or not GPU_USABLE
RUN_PREFIX = "QUICK-" if QUICK_TEST else ""

LOCAL_AUDIO = REPO / "data_audio" / ("_quick" if QUICK_TEST else "")
if WAV_ROOT:
    AUDIO_ROOT = Path(WAV_ROOT)
elif (DRIVE_ROOT is not None and not QUICK_TEST
      and (DRIVE_ROOT / "data_audio" / "wav").is_dir()):
    AUDIO_ROOT = DRIVE_ROOT / "data_audio"
else:
    AUDIO_ROOT = LOCAL_AUDIO
FEAT_DIR = LOCAL_AUDIO / "feat"           # always local: read at every training step

TRAINING_CONFIG = {
    "USE_CACHED_SCENES":  USE_CACHED_SCENES,
    "CACHED_SCENES_ROOT": CACHED_SCENES_ROOT,
    "ENCODER":            ENCODER,
    "EPOCH":              2 if QUICK_TEST else EPOCH,
    "BATCH_SIZE":         2 if QUICK_TEST else (MIN_BATCH_SIZE if USE_CPU else BATCH_SIZE),
    "LANG_NUM_MAX":       LANG_NUM_MAX,
    "LR":                 LR,
    "SEED":               SEED,
    "VAL_STEP":           5 if QUICK_TEST else VAL_STEP,
    "VERBOSE":            5 if QUICK_TEST else VERBOSE,
    "NUM_WORKERS":        0 if QUICK_TEST else NUM_WORKERS,
    "NUM_SCENES":         QUICK_SCENES if QUICK_TEST else -1,
    "NUM_VAL_SCENES":     QUICK_SCENES if QUICK_TEST else -1,
    "ONLY_CACHED_AUDIO":  QUICK_TEST,
    "EXTRA_ARGS":         EXTRA_ARGS,
    "CHECKPOINT":         CHECKPOINT_FILE,
    "RUN_PREFIX":         RUN_PREFIX,
    "CPU":                USE_CPU,
    "AUDIO_ROOT":         AUDIO_ROOT,
    "AUDIO_FEAT_DIR":     FEAT_DIR,
}

def as_env(value):
    """fusion_arms._env parses these back: lists comma-separated, bools as true/false."""
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (list, tuple)):
        return ",".join(str(v) for v in value)
    return str(value)

for name, value in TRAINING_CONFIG.items():
    os.environ[f"VGA_{name}"] = as_env(value)
    print(f"  VGA_{name:<20} = {as_env(value)}")

## 6 — the experiment registry

One declarative entry per experiment (`device`, `produces`, `needs`). The arm, sweep and intervention lists are read from the runner scripts, so this cell cannot drift from them; no analysis is reimplemented in the notebook.

In [ ]:
sys.path.insert(0, str(REPO))
from experiments.audio import fusion_arms as FA, interventions as IV, sweeps as SW

AU = "experiments/audio"
ARMS_PY, SWEEPS_PY, INTER_PY = f"{AU}/fusion_arms.py", f"{AU}/sweeps.py", f"{AU}/interventions.py"
ANALYSIS_PY, ROBUST_PY, MEASURE_PY = f"{AU}/analysis.py", f"{AU}/robustness.py", f"{AU}/measurements.py"

def has(pattern):
    """True when a path or glob exists; relative patterns are under the repository."""
    return bool(_glob.glob(str(REPO / pattern)))

def arm_file(arm, name="scores.p"):
    return f"outputs/*_{RUN_PREFIX}VGA-{arm.upper()}/{name}"

def tag_file(tag, name="scores.p"):
    return f"outputs/*_{RUN_PREFIX}{tag.upper()}/{name}"

def scored(arm):
    return (f"an evaluated '{arm}' run (phase A)", arm_file(arm))

def diagnosed(arm):
    return (f"diagnostics of an evaluated '{arm}' run (phase A)", arm_file(arm, "diagnostics.p"))

DATA_NEEDS = [
    ("the train feature cache (run_audio_encode.ipynb)", str(FEAT_DIR / ENCODER / "train.h5")),
    ("the val feature cache (run_audio_encode.ipynb)", str(FEAT_DIR / ENCODER / "val.h5")),
]
if USE_CACHED_SCENES:
    DATA_NEEDS.insert(0, ("the scene cache (unzip cached_scenes.zip)",
                          f"{CACHED_SCENES_ROOT}/meta.json"))
WAV_NEEDS = [("the WAVs (run_audio_encode.ipynb; set WAV_ROOT)", str(AUDIO_ROOT / "wav"))]
TRAINED_SPOT = [("a trained 'spot' run (phase A1)", arm_file("spot", "model_last.pth"))]

# Every experiment belongs to exactly one phase. Keeping them apart is what lets a Colab
# session that dies during phase A be resumed without re-running the rest.
PHASE_OF = {}          # stage key -> "A" .. "E"
RUN = {}               # stage key -> bool, assembled from the switches in section 1
EXPERIMENTS = []

def add(key, stage, device, script, args=(), produces=(), note="", needs=()):
    """`args` may be a function, for commands that depend on what has been produced."""
    if isinstance(script, tuple):
        script, *sub = script          # ("analysis.py", "slot-roles") -> subcommand
    else:
        sub = []
    EXPERIMENTS.append(dict(key=key, stage=stage, device=device, script=script, sub=sub,
                            args=args, produces=list(produces), note=note, needs=list(needs)))

def stage(key, phase, enabled):
    """Declare a stage, its phase and its switch, in one place."""
    PHASE_OF[key] = phase
    RUN[key] = bool(enabled)

# PHASE A -- training arms. One stage, one switch per arm.

for arm, cfg in FA.ARMS.items():
    stage(f"arm_{arm}", "A", RUN_ARMS.get(arm, False))
    add(f"train-{arm}", f"arm_{arm}", "gpu", (ARMS_PY, "run", arm, "--eval"),
        produces=[arm_file(arm)], note=cfg["note"], needs=DATA_NEEDS)

stage("seeds", "A", RUN_TRAIN_SEEDS)
add("train-seeds", "seeds", "gpu", (SWEEPS_PY, "run", "seeds", "--values", *SEEDS, "--eval"),
    produces=[tag_file(f"SWEEP-SEEDS-{s}") for s in SEEDS],
    note="the proposed model under more seeds -- the noise floor every gap is read against",
    needs=DATA_NEEDS)

# PHASE B -- evaluation only.

stage("eval_arms", "B", RUN_EVAL_ARMS)
add("eval-arms", "eval_arms", "gpu", (ARMS_PY, "eval-all"),
    note="score every trained arm that has no scores yet")

stage("controls", "B", RUN_CONTROLS)
add("controls", "controls", "gpu", (INTER_PY, "controls"),
    produces=[arm_file("spot", f"interventions/{name}/scores.p") for name in IV.CONTROLS],
    note="replace the audio at test time: " + ", ".join(IV.CONTROLS), needs=TRAINED_SPOT)

stage("knockout", "B", RUN_KNOCKOUT)
add("knockout", "knockout", "gpu", (INTER_PY, "knockout"),
    produces=[arm_file("spot", f"interventions/{name}/scores.p") for name in IV.KNOCKOUTS],
    note="switch one component off at test time: " + ", ".join(IV.KNOCKOUTS),
    needs=TRAINED_SPOT)

# PHASE C -- sweeps. The default point of each is the 'spot' arm and is not retrained.

def sweep_values(name, with_default=False):
    """--values for a sweep, or [] to run its full grid."""
    spec, chosen = SW.SWEEPS[name], SWEEP_VALUES.get(name)
    if not chosen or "env_grid" in spec:
        return []
    keep = [v for v in spec["values"] if v in chosen or (with_default and v == spec["default"])]
    return ["--values", *keep]

SWEEP_NAMES = [name for name in SW.SWEEPS if name != "seeds"]
for name in SWEEP_NAMES:
    spec = SW.SWEEPS[name]
    stage(f"sweep_{name}", "C", RUN_SWEEPS.get(name, False))
    add(f"sweep-{name}", f"sweep_{name}", "gpu",
        (SWEEPS_PY, "run", name, *sweep_values(name), "--eval"),
        note=spec["question"].split(". ")[0] + ".",
        needs=DATA_NEEDS + (WAV_NEEDS if spec.get("needs_cache") else []))

# PHASE D -- tables and analyses.

stage("tables", "D", RUN_TABLES)
add("table-arms", "tables", "cpu", (ARMS_PY, "table"),
    note="progressive and leave-one-out tables, with deltas against 'spot'",
    needs=[scored("spot")])
add("table-interventions", "tables", "cpu", (INTER_PY, "table"),
    note="controls and knockouts against the clean score of the same checkpoint",
    needs=[scored("spot")])
add("table-seeds", "tables", "cpu",
    (SWEEPS_PY, "table", "seeds", "--values", SEED, *SEEDS),
    note="mean and standard deviation over seeds", needs=[scored("spot")])
for name in SWEEP_NAMES:
    if RUN_SWEEPS.get(name, False):
        add(f"table-sweep-{name}", "tables", "cpu",
            (SWEEPS_PY, "table", name, *sweep_values(name, with_default=True)),
            note=f"the '{name}' sweep", needs=[scored("spot")])

stage("analyses", "D", RUN_ANALYSES)
add("results-table", "analyses", "cpu", (ANALYSIS_PY, "results-table"),
    args=lambda: ["--folders", *[a for a in FA.ARMS if has(arm_file(a))]],
    note="the metric table for every evaluated arm, beside the published rows",
    needs=[scored("spot")])
add("slot-roles", "analyses", "cpu",
    (ANALYSIS_PY, "slot-roles", "--folder", "spot", "--by_outcome"),
    note="did the slots take distinct spans, and did the role election settle?",
    needs=[diagnosed("spot")])
add("ot-sharpness", "analyses", "cpu", (ANALYSIS_PY, "ot-sharpness"),
    args=lambda: ["--folders", *[a for a in ("spot", "spot-softmax", "spot-k2")
                                 if has(arm_file(a, "diagnostics.p"))]],
    note="margin between the best and second-best proposal in the transport plan",
    needs=[diagnosed("spot")])
add("error-taxonomy", "analyses", "cpu", (ANALYSIS_PY, "error-taxonomy", "--folder", "spot"),
    note="object class misheard vs wrong instance picked", needs=[diagnosed("spot")])
add("gain-attribution", "analyses", "cpu",
    (ANALYSIS_PY, "gain-attribution", "--folders", "xattn", "spot"),
    note="where the gain over attentive fusion comes from, paired per utterance",
    needs=[diagnosed("spot"), diagnosed("xattn")])
add("complexity", "analyses", "cpu",
    (ANALYSIS_PY, "complexity", "--folder", "spot", "--baseline", "xattn"),
    note="accuracy by utterance duration and scene clutter",
    needs=[diagnosed("spot"), diagnosed("xattn")])
add("significance-softmax", "analyses", "cpu",
    (ANALYSIS_PY, "significance", "spot-softmax", "spot"),
    note="paired permutation test for the central claim",
    needs=[scored("spot"), scored("spot-softmax")])
add("significance-xattn", "analyses", "cpu", (ANALYSIS_PY, "significance", "xattn", "spot"),
    note="paired permutation test against attentive fusion",
    needs=[scored("spot"), scored("xattn")])
add("seed-aggregate", "analyses", "cpu", (ANALYSIS_PY, "seed-aggregate"),
    args=lambda: ["--folders", "spot", *[f"SWEEP-SEEDS-{s}" for s in SEEDS
                                         if has(tag_file(f"SWEEP-SEEDS-{s}"))]],
    note="mean and sd over the seed runs, with the 2-sd threshold a gap must clear",
    needs=[scored("spot"), ("at least one extra seed (phase A5)", tag_file("SWEEP-SEEDS-*"))])

stage("cost", "D", RUN_COST)
add("params", "cost", "cpu", (MEASURE_PY, "params"), note="parameters by module")
add("latency", "cost", "cpu", (MEASURE_PY, "latency", "--device", "cpu" if USE_CPU else "cuda"),
    note="forward-pass time of the acoustic branch and fusion head, synthetic tensors")

# PHASE E -- robustness.

stage("speaker_ood", "E", RUN_SPEAKER_OOD)
add("speaker-ood", "speaker_ood", "cpu",
    (ROBUST_PY, "speaker-ood", "--folder", "spot", "--per_voice"),
    note="val accuracy on voices seen in training vs voices held out",
    needs=[diagnosed("spot"),
           ("the synthesis manifests (run_audio_encode.ipynb)", str(AUDIO_ROOT / "manifest_val.json"))])

stage("noise", "E", RUN_NOISE)
add("noise", "noise", "gpu",
    (ROBUST_PY, "noise", "--folder", "spot", "--snr", *NOISE_SNR, "--run"),
    produces=[arm_file("spot", f"noise/snr{snr:g}/scores.p") for snr in NOISE_SNR],
    note="white noise added to the waveform before the frozen encoder",
    needs=TRAINED_SPOT + WAV_NEEDS)

stage("asr_wer", "E", RUN_ASR_WER)
add("asr-transcribe", "asr_wer", "gpu", (ROBUST_PY, "asr-cascade", "--build"),
    produces=[str(AUDIO_ROOT / "asr" / "transcripts_val.json")],
    note="transcribe the val WAVs with a CTC recogniser", needs=WAV_NEEDS)
add("asr-wer", "asr_wer", "cpu", (ROBUST_PY, "asr-cascade", "--wer"),
    note="word-error rate of those transcripts -- the cascade's error floor",
    needs=[("the transcripts (asr-transcribe)", str(AUDIO_ROOT / "asr" / "transcripts_val.json"))])

print(f"{len(EXPERIMENTS)} experiments in {len(PHASE_OF)} stages")

## 7 — preflight

What will happen, before it happens. `action` column: `RUN`, `SKIP-off`, `SKIP-done`, `SKIP-gpu`, `SKIP-input`. An analysis that waits on a run from an earlier phase shows `SKIP-input` here and is decided again when its cell runs.

In [ ]:
def produced(exp):
    """True when every declared output already exists (globs allowed)."""
    if not exp["produces"]:
        return False
    return all(has(pattern) for pattern in exp["produces"])

def missing_inputs(exp):
    for label, pattern in exp["needs"]:
        if not has(pattern):
            return label
    return None

def decide(exp):
    if not RUN.get(exp["stage"], False):
        return "SKIP-off", "switched off in the config cell"
    gap = missing_inputs(exp)
    if gap:
        return "SKIP-input", f"needs {gap}"
    if exp["device"] == "gpu" and not GPU_USABLE and not FORCE_GPU:
        return "SKIP-gpu", "no usable GPU (set FORCE_GPU=True to force)"
    if produced(exp) and not OVERRIDE_EXP:
        return "SKIP-done", "outputs already exist (set OVERRIDE_EXP=True to redo)"
    return "RUN", ""

PHASE_NAME = {"A": "training arms", "B": "evaluation only", "C": "sweeps",
              "D": "tables and analyses", "E": "robustness"}
PHASE_TITLE = {
    "A": "TRAINING ARMS    -- GPU. One run per arm, evaluated as it finishes.",
    "B": "EVALUATION ONLY  -- GPU, minutes. Needs a trained 'spot' run. Trains nothing.",
    "C": "SWEEPS           -- GPU. One setting varied; the default point is the 'spot' arm.",
    "D": "TABLES, ANALYSES -- CPU. Reads files off disk.",
    "E": "ROBUSTNESS       -- unseen speakers (CPU), noise and ASR error rate (GPU).",
}

PLAN = {exp["key"]: decide(exp)[0] for exp in EXPERIMENTS}

for phase in ("A", "B", "C", "D", "E"):
    chosen = [e for e in EXPERIMENTS if PHASE_OF[e["stage"]] == phase]
    print(f"\n{'=' * 100}")
    print(f"PHASE {phase} -- {PHASE_TITLE[phase]}")
    print("=" * 100)
    print(f"  {'experiment':<24} {'dev':<4} {'action':<11} why")
    print("  " + "-" * 96)
    for exp in chosen:
        action, why = decide(exp)
        if action == "SKIP-off" and phase in ("A", "C"):
            continue                    # a long list of arms that are simply switched off
        print(f"  {exp['key']:<24} {exp['device']:<4} {action:<11} {why}")
    running = sum(1 for e in chosen if PLAN[e["key"]] == "RUN")
    print(f"  -> {running} of {len(chosen)} will run")

print(f"\n{'=' * 100}")
counts = {a: sum(1 for v in PLAN.values() if v == a) for a in sorted(set(PLAN.values()))}
print("TOTAL   " + "   ".join(f"{a}: {n}" for a, n in counts.items()))
trainings = sum(1 for e in EXPERIMENTS if PLAN[e["key"]] == "RUN" and e["key"].startswith("train-"))
print(f"\ntraining experiments that will run: {trainings}"
      + (f" (train-seeds is {len(SEEDS)} runs)" if PLAN.get("train-seeds") == "RUN" else ""))
if FORCE_GPU and not GPU_USABLE:
    print(f"\n  FORCE_GPU is on without a GPU -- GPU experiments will use "
          f"batch_size={MIN_BATCH_SIZE} on CPU.")

## 8 — run

One cell per stage. Each experiment writes a report to `outputs/reports/<experiment>/` (log, `result.json`, `report.md`); a non-zero exit is recorded and the run continues.

In [ ]:
# The report exists because a Colab runtime is disposable: the scrollback is gone the
# moment the tab closes, and the exit code alone does not say what a run was for.
#
# run_single(key)  -- one experiment, one cell. Used for the training arms, so a run can
#                     be started, skipped or resumed on its own.
# run_stage(stage) -- every experiment in a stage; used where members are cheap and
#                     always wanted together (tables, analyses).

REPORTS = REPO / "outputs" / "reports"

def command_args(exp):
    extra = exp["args"]() if callable(exp["args"]) else list(exp["args"])
    return [str(a) for a in (*exp["sub"], *extra)]

def describe(exp):
    """The header printed before a command runs."""
    device = "GPU" if exp["device"] == "gpu" else "CPU"
    produces = exp["produces"] or ["(no declared output file)"]
    phase = PHASE_OF.get(exp["stage"], "?")
    lines = [
        "=" * 88,
        f"EXPERIMENT   {exp['key']}",
        f"phase        {phase} -- {PHASE_NAME.get(phase, '')}   |   requires {device}",
        f"purpose      {textwrap.shorten(exp['note'] or '--', 200)}",
        f"produces     {produces[0]}",
    ]
    lines += [f"             {p}" for p in produces[1:4]]
    return "\n".join(lines)

def write_report(exp, record, log_text):
    """One folder per experiment: raw log, structured result, and a readable summary."""
    folder = REPORTS / exp["key"]
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "log.txt").write_text(log_text)

    phase = PHASE_OF.get(exp["stage"], "?")
    record = dict(record)
    record.update(key=exp["key"], stage=exp["stage"], phase=phase, device=exp["device"],
                  purpose=exp["note"], script=exp["script"], produces=exp["produces"],
                  finished_at=time.strftime("%Y-%m-%d %H:%M:%S"),
                  environment="Google Colab" if IN_COLAB else "local", gpu=GPU_NAME or "none")
    (folder / "result.json").write_text(json.dumps(record, indent=2))

    status = ("SUCCESS" if record.get("exit_code") == 0
              else f"FAILED (exit {record.get('exit_code')})")
    outputs_present = [f"  - {p}  {'OK' if has(p) else 'MISSING'}"
                       for p in exp["produces"]] or ["  (none declared)"]
    tail = log_text.strip().splitlines()[-40:]
    (folder / "report.md").write_text("\n".join([
        f"# {exp['key']}", "",
        f"**{status}** in {record.get('elapsed_sec', 0)}s "
        f"on {record['finished_at']} ({record['environment']}, GPU: {record['gpu']})", "",
        f"Phase {phase} ({PHASE_NAME.get(phase, '')}).", "",
        "## What this experiment tests", "", exp["note"] or "_no description recorded_", "",
        "## How it was run", "", "```", record.get("cmd", ""), "```", "",
        "## Declared outputs", "", *outputs_present, "",
        "## Last lines of output", "", "```", *tail, "```", "",
        "_Full output in `log.txt`; structured fields in `result.json`._",
    ]))
    return folder

def run_experiment(exp):
    action, why = decide(exp)           # decided now, not at preflight: inputs may have appeared
    PLAN[exp["key"]] = action
    if action != "RUN":
        RESULTS[exp["key"]] = {"action": action}
        print(f"[{exp['key']}] {action}  ({why})")
        return

    cmd = [sys.executable, exp["script"], *command_args(exp)]
    printable = " ".join(cmd)
    print("\n" + describe(exp))
    print(f"$ {printable}\n{'-' * 88}")

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = GPU_ID if (DEVICE == "cuda" and GPU_USABLE) else ""
    env["PYTHONUNBUFFERED"] = "1"

    started = time.time()
    captured = []
    process = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        captured.append(line)
    code_ = process.wait()
    elapsed = time.time() - started

    record = {"action": "RUN", "exit_code": code_, "elapsed_sec": round(elapsed, 1),
              "cmd": printable}
    RESULTS[exp["key"]] = record

    if code_ == 0:
        verdict = "OK"
    elif code_ in (247, -9, 137):
        # 247/-9/137 are all SIGKILL: the host ran out of RAM, not the GPU.
        verdict = f"FAILED (exit {code_}) -- killed by the OS, out of host RAM. Lower NUM_WORKERS."
    else:
        verdict = f"FAILED (exit {code_})"

    folder = write_report(exp, record, "".join(captured))
    print(f"{'-' * 88}\n[{exp['key']}] {verdict} in {elapsed:.1f}s")
    print(f"           report -> {folder.relative_to(REPO)}/report.md")
    mirror_to_drive(quiet=True)

def _banner(title, subtitle=""):
    print(f"\n{'#' * 88}")
    print(f"# {title}")
    if subtitle:
        for line in textwrap.wrap(subtitle, 84):
            print(f"# {line}")
    print("#" * 88)

def run_single(key):
    """Run exactly one experiment. This is the phase-A entry point."""
    matches = [e for e in EXPERIMENTS if e["key"] == key]
    if not matches:
        print(f"[{key}] no such experiment -- check the key against the registry cell.")
        return
    exp = matches[0]
    _banner(f"{key}   (phase {PHASE_OF.get(exp['stage'], '?')})", exp["note"])
    run_experiment(exp)

def run_stage(stage_key):
    """Run every experiment in a stage."""
    chosen = [e for e in EXPERIMENTS if e["stage"] == stage_key]
    _banner(f"STAGE: {stage_key}   (phase {PHASE_OF.get(stage_key, '?')}; "
            f"{len(chosen)} experiment(s))")
    for exp in chosen:
        run_experiment(exp)
    tally = {}
    for exp in chosen:
        action = RESULTS.get(exp["key"], {}).get("action", "SKIP")
        tally[action] = tally.get(action, 0) + 1
    print(f"\n# stage '{stage_key}' finished: "
          + "  ".join(f"{k}={v}" for k, v in sorted(tally.items())))

print("runner ready -- reports go to outputs/reports/<experiment>/")

---
# Phase A — training arms

GPU. Every arm trains the acoustic branch and the fusion head only, on the same cached proposals and the same cached speech features, and is evaluated as soon as it finishes. Each differs from `spot` in exactly one thing. Keep A1 in any subset — every other arm is read against it.

### A1 — `spot`, the proposed model (reference arm)

**Ablates:** nothing. Every delta in the tables, every evaluation-only experiment in phase B and the default point of every sweep is this run.

In [ ]:
# switch: RUN_ARMS["spot"]
run_single("train-spot")

### A2 — baselines: `concat` and `xattn`

**`concat`** reproduces AP-Refer's fusion (mean-pooled audio, concatenated, MLP) inside this pipeline, so the comparison shares a detector and a corpus. **`xattn`** is attentive fusion without transport. Together with `spot` they are the progressive table.

In [ ]:
# switches: RUN_ARMS["concat"], RUN_ARMS["xattn"]
run_single("train-concat")
run_single("train-xattn")

### A3 — `spot-softmax`, the central claim

**Ablates:** the transport constraint only. Sinkhorn becomes a row-wise softmax; the cost matrix, temperature, slot masses, read-out and parameter count are unchanged. The method predicts the drop falls on the **Multiple** subset — read Unique and Multiple separately.

In [ ]:
# switch: RUN_ARMS["spot-softmax"]
run_single("train-spot-softmax")

### A4 — `spot-norsk` and `spot-noprosody`

**Ablate:** the relation-slot geometric kernel (replaced by the fixed distance prior) and the prosody stream — the two components the method adds on the acoustic side.

In [ ]:
# switches: RUN_ARMS["spot-norsk"], RUN_ARMS["spot-noprosody"]
run_single("train-spot-norsk")
run_single("train-spot-noprosody")

### A5 — seed repeats

Repeats `spot` under the seeds in `SEEDS` so every gap can be reported against run-to-run spread. Seed 42 is the A1 run and is not trained again.

In [ ]:
# switch: RUN_TRAIN_SEEDS
run_single("train-seeds")

### A6 — extended leave-one-out arms

The remaining arms, off by default: two slots only, no bidirectional update, fixed roles, no spatial graph, no read-out gate, no frame residual, no copy-paste, no time prior, no SpecAugment. Runs the ones switched on in `RUN_ARMS`.

In [ ]:
# switches: the "extended" entries of RUN_ARMS
EXTENDED = [arm for arm in FA.ARMS if arm not in FA.CORE]
chosen = [arm for arm in EXTENDED if RUN_ARMS.get(arm, False)]
if not chosen:
    print("no extended arm is switched on:", ", ".join(EXTENDED))
for arm in chosen:
    run_single(f"train-{arm}")

---
# Phase B — evaluation only

**GPU, minutes.** Re-scores the trained `spot` checkpoint with something changed at test time. Nothing is trained, and results are archived under `outputs/<run>/interventions/<name>/`, so the run's own scores are never overwritten.

### B1 — score trained arms

Phase A evaluates each arm as it finishes; this catches any arm that trained but was not scored (an interrupted session).

In [ ]:
# switch: RUN_EVAL_ARMS
run_stage("eval_arms")

### B2 — controls: is the model listening?

The scene, the proposals and the labels stay; only the audio changes.

| control | audio at test time |
|---|---|
| `silence` | zeros |
| `mismatch` | an utterance about a different object class |
| `reverse` | the same frames in reverse order |
| `first-half` / `second-half` | one half of the clip |

`silence` and `mismatch` are the floor — what the scene alone reaches. Accuracy above that floor is what the speech contributes.

In [ ]:
# switch: RUN_CONTROLS
run_stage("controls")

### B3 — knockouts: what does the trained model rely on?

One component switched off at test time, weights untouched: transport routing, the relation kernel, the bidirectional update, the read-out gate, the frame residual, the spatial graph, the time prior, and each prosodic cue group (flattened to its per-utterance mean). A phase-A arm shows whether a component is *needed*; a knockout shows whether the trained model *uses* it.

In [ ]:
# switch: RUN_KNOCKOUT
run_stage("knockout")

---
# Phase C — sweeps

GPU, off by default. One setting varied with everything else at its default. `SWEEP_VALUES` holds the lighter grids; the default point of every sweep is the A1 run and is not trained again. `encoder_layer`, `encoder` and `frames` re-encode the WAVs once per point.

In [ ]:
# switches: RUN_SWEEPS
chosen = [name for name in SWEEP_NAMES if RUN_SWEEPS.get(name, False)]
if not chosen:
    print("no sweep is switched on:", ", ".join(SWEEP_NAMES))
for name in chosen:
    run_stage(f"sweep_{name}")

---
# Phase D — tables and analyses

CPU, seconds. Reads `scores.p`, `predictions.p` and `diagnostics.p` from the runs above; builds no model. Each table is also saved in `outputs/reports/<experiment>/log.txt`.

### D1 — tables

| experiment | shows |
|---|---|
| `table-arms` | progressive and leave-one-out tables, deltas against `spot` |
| `table-interventions` | controls and knockouts against the clean score |
| `table-seeds` | mean ± sd over seeds |
| `table-sweep-<name>` | each sweep that is switched on |

In [ ]:
# switch: RUN_TABLES
run_stage("tables")

### D2 — analyses

| experiment | answers |
|---|---|
| `results-table` | all evaluated arms beside the published rows |
| `slot-roles` | did the slots separate, and did the role election settle on a slot? |
| `ot-sharpness` | does the transport plan separate the best proposal from the runner-up? |
| `error-taxonomy` | class misheard, or wrong instance picked? |
| `gain-attribution`, `complexity` | where the gain over attentive fusion comes from |
| `significance-*` | paired permutation test and bootstrap interval |
| `seed-aggregate` | the 2-sd threshold a gap has to clear |

In [ ]:
# switch: RUN_ANALYSES
run_stage("analyses")

### D3 — cost

Parameters by module, and forward-pass time of the acoustic branch and fusion head on synthetic tensors.

In [ ]:
# switch: RUN_COST
run_stage("cost")

---
# Phase E — robustness

`speaker-ood` needs no retraining and no GPU: it splits the val score by whether the voice was heard in training. `noise` and the ASR error rate read the WAVs.

In [ ]:
# switches: RUN_SPEAKER_OOD, RUN_NOISE, RUN_ASR_WER
run_stage("speaker_ood")
run_stage("noise")
run_stage("asr_wer")

## Summary

What ran, what it produced, and what's outstanding — grouped by phase.

In [ ]:
print("=" * 96)
print("EXECUTION SUMMARY")
print("=" * 96)

ok = failed = skipped = 0
for phase in ("A", "B", "C", "D", "E"):
    chosen = [e for e in EXPERIMENTS if PHASE_OF[e["stage"]] == phase]
    shown = [e for e in chosen if e["key"] in RESULTS
             and RESULTS[e["key"]].get("action") != "SKIP-off"]
    if not shown:
        continue
    print(f"\nPHASE {phase} -- {PHASE_NAME[phase]}")
    for exp in shown:
        record = RESULTS[exp["key"]]
        if record.get("action") != "RUN":
            skipped += 1
            status = record.get("action", "SKIP")
        elif record.get("exit_code") == 0:
            ok += 1
            status = f"OK      {record['elapsed_sec']:>8.1f}s"
        else:
            failed += 1
            status = f"FAILED  exit {record.get('exit_code')}"
        print(f"  {exp['key']:<26} {status}")

print(f"\n{ok} succeeded, {failed} failed, {skipped} skipped")
print(f"reports: {REPORTS.relative_to(REPO)}/<experiment>/report.md")
if failed:
    print("\nfailed experiments -- the last lines of each log are in its report.md:")
    for key, record in RESULTS.items():
        if record.get("action") == "RUN" and record.get("exit_code") != 0:
            print(f"  {key}")
mirror_to_drive()